# Day 060 — Exercise 2: Cache-Aside Pattern

**Cache-aside** (also called *lazy loading*) is the most common caching pattern:

1. Check the cache — if found, return it (**cache hit**)
2. If not found (**cache miss**): call the slow function, store the result, return it

The cache is *populated on demand* — only entries that are actually requested get cached. This avoids pre-loading data that may never be used.

In [ ]:
# --- Provided: SimpleCache (from Exercise 1) ---
import time
from typing import Any

class SimpleCache:
    def __init__(self):
        self._store: dict = {}

    def set(self, key: str, value: Any, ttl: float = 60.0) -> None:
        self._store[key] = (value, time.monotonic() + ttl)

    def get(self, key: str) -> Any:
        entry = self._store.get(key)
        if entry is None:
            return None
        value, expires_at = entry
        if time.monotonic() > expires_at:
            del self._store[key]
            return None
        return value

    def has(self, key: str) -> bool:
        return self.get(key) is not None

    def clear(self) -> int:
        n = len(self._store)
        self._store.clear()
        return n

    def __len__(self) -> int:
        now = time.monotonic()
        return sum(1 for _, exp in self._store.values() if now <= exp)


In [ ]:
from typing import Any, Callable


## Task

Implement `cache_aside(key, cache, fn, ttl=60.0) -> Any`:

- If `cache.has(key)`: return `cache.get(key)`
- Otherwise: `value = fn()`, then `cache.set(key, value, ttl)`, return `value`
- `fn` is a zero-argument callable (`lambda: expensive_call()`)

## Your Implementation

In [ ]:
def cache_aside(key: str, cache: SimpleCache, fn: Callable[[], Any],
                ttl: float = 60.0) -> Any:
    """Cache-aside pattern: check cache first, call fn() on miss.

    1. If cache.has(key): return cache.get(key)
    2. Else: call fn(), store result with ttl, return result
    fn is called with no arguments.
    """
    # TODO: check cache.has(key), call fn() on miss, cache the result
    raise NotImplementedError


In [ ]:
def cache_aside(key, cache, fn, ttl=60.0):
    if cache.has(key):
        return cache.get(key)
    value = fn()
    cache.set(key, value, ttl)
    return value


## Automated checks

In [ ]:
score, total = 0, 4
try:
    call_log = {"n": 0}

    def expensive():
        call_log["n"] += 1
        return f"result_{call_log['n']}"

    cache = SimpleCache()

    # first call — miss, fn invoked
    r1 = cache_aside("k", cache, expensive, ttl=10.0)
    assert r1 == "result_1", f"Got {r1}"
    assert call_log["n"] == 1
    score += 1; print("\u2705 first call invokes fn and returns result")

    # second call — hit, fn NOT invoked again
    r2 = cache_aside("k", cache, expensive, ttl=10.0)
    assert r2 == "result_1", f"Got {r2}"
    assert call_log["n"] == 1, f"fn called {call_log['n']} times, expected 1"
    score += 1; print("\u2705 second call returns cached result (fn not called)")

    # after TTL expires — miss again
    cache2 = SimpleCache()
    r3 = cache_aside("k2", cache2, expensive, ttl=0.02)
    assert call_log["n"] == 2
    time.sleep(0.06)
    r4 = cache_aside("k2", cache2, expensive, ttl=0.02)
    assert call_log["n"] == 3, f"fn should have been called again after TTL, got {call_log['n']}"
    score += 1; print("\u2705 after TTL expiry, fn is called again")

    # different keys are cached independently
    cache3 = SimpleCache()
    cache_aside("a", cache3, lambda: "aa", ttl=10.0)
    cache_aside("b", cache3, lambda: "bb", ttl=10.0)
    assert cache3.get("a") == "aa" and cache3.get("b") == "bb"
    score += 1; print("\u2705 different keys cached independently")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def cache_aside(key, cache, fn, ttl=60.0):
    if cache.has(key):
        return cache.get(key)
    value = fn()
    cache.set(key, value, ttl)
    return value
```

**Why `has()` instead of `get() is not None`?** If `fn()` could return `None` legitimately, `get() is not None` would treat a cached `None` as a miss and re-call `fn()`. Using `has()` correctly distinguishes 'key present' from 'key absent'.

</details>